# 04 · Does Beating the Spread Carry Over?

**Goal:** Test whether a team that beat the point spread in its last game is more or less
likely to beat the spread in its next game.

**Data:** `nflmodel.data.read_games()`, filtered to 2012–2025 played games

**Contents:**
1. Load
2. Build the team-game table (one row per team per game)
3. Attach each team's previous game
4. Main test: cover rate by previous cover
5. Exploratory: correlation and blowout buckets
6. Robustness: home-team rows only; 2012–2021 vs. 2022–2025
7. Findings and decisions

**Why it matters:** This is a small test of market efficiency. The point spread
should already account for everything known before kickoff, including last week's
result. If past performance against the spread predicts future performance, the
market is systematically over- or under-reacting, and that would be a candidate
feature for the Phase 2 margin model. If it doesn't, the feature hasn't earned
its place.

## Hypotheses (stated before looking at results)

1. **Efficient market (null):** last game's result against the spread has no
   relationship with this game's. Cover rates stay near 50% either way.
2. **Overreaction:** teams that just beat the spread get overrated, so their next
   line is too generous and they cover *less* often than 50%.
3. **Momentum:** teams that just beat the spread are improving faster than the
   market adjusts, so they cover *more* often than 50%.

**Main test (decided in advance):** cover rate in a team's next game, split by
whether it covered its previous game. All other cuts are exploratory.

**Benchmarks for judging results:**
- Statistical: with ~3,500 team-games per group, differences under ~2 percentage
  points are within noise.
- Financial: at standard −110 odds, a strategy must win 52.4% of bets to break even.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from nflmodel.data import read_games

## 1. Load

In [2]:
# File covers 2005-2026 incl. unplayed games; keep the original 2012-2025 played set
clean = read_games()
clean = clean[clean["season"].between(2012, 2025) & clean["home_score"].notna()].reset_index(drop=True)
clean.shape

(3829, 21)

## 2. Build the team-game table

In [3]:
home = pd.DataFrame({
    "game_id": clean["game_id"],
    "season": clean["season"],
    "week": clean["week"],
    "kickoff": clean["kickoff"],
    "is_playoff": clean["is_playoff"], 
    "team": clean["home_team"],
    "opponent": clean["away_team"],
    "is_home": True,
    "margin": clean["result"],                  # team's points minus opponent's
    "expected_margin": clean["spread_line"]     # market's expected margin for this team
})

In [4]:
away = pd.DataFrame({
    "game_id": clean["game_id"],
    "season": clean["season"],
    "week": clean["week"],
    "kickoff": clean["kickoff"],
    "is_playoff": clean["is_playoff"], 
    "team": clean["away_team"],
    "opponent": clean["home_team"],
    "is_home": False,
    "margin": -clean["result"],
    "expected_margin": -clean["spread_line"]
})

In [5]:
tg = pd.concat([home, away], ignore_index = True)
tg = tg.sort_values(["team", "kickoff"]).reset_index(drop=True)

# How much the team beat the spread by, from its own side
tg["ats_margin"] = tg["margin"] - tg["expected_margin"]

# 1 = covered, 0 = didn't, NaN = push (landed exactly on the spread)
tg["covered"] = (tg["ats_margin"] > 0).astype(float).where(tg["ats_margin"] != 0)

tg.head(20)

,game_id,season,week,kickoff,is_playoff,team,opponent,is_home,margin,expected_margin,ats_margin,covered
0,2012_01_SEA_ARI,2012,1,2012-09-09 16:25:00,False,ARI,SEA,True,4,-1.0,5.0,1.0
1,2012_02_ARI_NE,2012,2,2012-09-16 13:00:00,False,ARI,NE,False,2,-13.0,15.0,1.0
2,2012_03_PHI_ARI,2012,3,2012-09-23 16:05:00,False,ARI,PHI,True,21,-3.0,24.0,1.0
3,2012_04_MIA_ARI,2012,4,2012-09-30 16:05:00,False,ARI,MIA,True,3,4.0,-1.0,0.0
4,2012_05_ARI_STL,2012,5,2012-10-04 20:20:00,False,ARI,LA,False,-14,1.0,-15.0,0.0
5,2012_06_BUF_ARI,2012,6,2012-10-14 16:05:00,False,ARI,BUF,True,-3,5.0,-8.0,0.0
6,2012_07_ARI_MIN,2012,7,2012-10-21 13:00:00,False,ARI,MIN,False,-7,-7.0,0.0,NaN
7,2012_08_SF_ARI,2012,8,2012-10-29 20:30:00,False,ARI,SF,True,-21,-7.0,-14.0,0.0
8,2012_09_ARI_GB,2012,9,2012-11-04 13:00:00,False,ARI,GB,False,-14,-10.5,-3.5,0.0
9,2012_11_ARI_ATL,2012,11,2012-11-18 13:00:00,False,ARI,ATL,False,-4,-9.5,5.5,1.0


In [6]:
# 1. Every game appears exactly twice
assert len(tg) == 2 * len(clean), "Row count should double"
assert (tg["game_id"].value_counts() == 2).all(), "Some game isn't exactly twice"

# 2. The two sides mirror each other: ATS margins in a game sum to zero
ats_sum = tg.groupby("game_id")["ats_margin"].sum()
assert (ats_sum.abs() < 1e-9).all(), "Home and away ATS margins don't mirror"

# 3. In every non-push game, exactly one team covered
cov_sum = tg.dropna(subset=["covered"]).groupby("game_id")["covered"].sum()
assert (cov_sum == 1).all(), "Both or neither team covered in some game"

print("All checks passed")
print(f"Team-games: {len(tg)}, pushes: {tg['covered'].isna().sum() // 2} games")

All checks passed
Team-games: 7658, pushes: 93 games


In [7]:
tg[(tg["team"] == "CAR") & (tg["season"] == 2023)][
    ["week", "opponent", "is_home", "margin", "expected_margin", "ats_margin", "covered"]
].sort_values("week")

,week,opponent,is_home,margin,expected_margin,ats_margin,covered
1142,1,ATL,False,-14,-3.5,-10.5,0.0
1143,2,NO,True,-3,-3.0,0.0,NaN
1144,3,SEA,False,-10,-5.0,-5.0,0.0
1145,4,MIN,True,-8,-4.5,-3.5,0.0
1146,5,DET,False,-18,-9.5,-8.5,0.0
1147,6,MIA,False,-21,-14.0,-7.0,0.0
1148,8,HOU,True,2,-3.0,5.0,1.0
1149,9,IND,True,-14,-1.5,-12.5,0.0
1150,10,CHI,False,-3,-3.0,0.0,NaN
1151,11,DAL,True,-23,-11.0,-12.0,0.0


From the table above, we can see that the Panthers were terrible in 2023